# 02 — Pandas approfondi

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- maîtriser `Series` et `DataFrame` et leurs principales opérations;
- lire et écrire CSV, Parquet, Excel, JSON;
- pratiquer l'**indexing** avancé (`loc`, `iloc`, `query`, `isin`);
- comprendre et utiliser un `MultiIndex`;
- grouper, agréger, pivoter (`groupby`, `pivot_table`, `melt`);
- manipuler des **séries temporelles** (resample, rolling, freq);
- joindre des tables (`merge`, `concat`, `join`);


## Prérequis — ce que vous devez maîtriser

Ce notebook fait partie des **Side tracks**. Il est conçu pour être autonome, à condition d'avoir suivi le niveau **Intermédiaire** de la formation principale. Vous devez donc être à l'aise avec :

- les fondamentaux Python (types, fonctions typées, comprehensions);
- NumPy de base — un `DataFrame` est construit sur des `ndarray`;
- la notion de table SQL (lignes, colonnes, jointures, group by);

Ce que ce notebook **ne couvrira pas** (ou seulement en mention) :

- la visualisation (chapitre Matplotlib/Seaborn);
- Polars (voir `03_Polars_complet.ipynb`) — alternative plus moderne;
- les gros volumes distribués (Dask, Spark);


> **Note.** Pandas reste la bibliothèque de data wrangling la plus installée au monde. Elle est parfois lente sur de très gros volumes (> 1 Go) : dans ce cas, Polars ou DuckDB sont à considérer.


## Plan

1. Installation et import
2. Series : tableau 1D étiqueté
3. DataFrame : tableau 2D étiqueté
4. Lecture/écriture de fichiers
5. Inspection rapide
6. Sélection avec `[]`, `loc`, `iloc`
7. Filtrage et `query`
8. Ajout et suppression de colonnes
9. Tri et classement
10. Valeurs manquantes (`NaN`)
11. `groupby` et agrégations
12. `pivot_table` et `melt`
13. Jointures : `merge`, `concat`, `join`
14. MultiIndex
15. Séries temporelles
16. Apply et vectorisation
17. Synthèse
18. Exercices


---


## 1. Installation et import

Pandas s'installe avec `pip install pandas`. Son import canonique est :


In [ ]:
import pandas as pd
import numpy as np
pd.__version__

---


## 2. `Series` — tableau 1D étiqueté

Une `Series` est une colonne : un tableau 1D de valeurs (comme un `ndarray`) avec un **index** qui étiquette chaque élément.


In [ ]:
s = pd.Series([10, 20, 30, 40])
s

In [ ]:
s = pd.Series([10, 20, 30], index=['a', 'b', 'c'])
s

In [ ]:
s['b']

In [ ]:
s.values  # renvoie un ndarray

In [ ]:
s.index

Une `Series` a un `dtype` comme un `ndarray` :


In [ ]:
s.dtype

---


## 3. `DataFrame` — tableau 2D étiqueté

Un `DataFrame` est l'objet principal de Pandas : une collection de `Series` partageant le même index.


In [ ]:
df = pd.DataFrame({
    'nom': ['Alice', 'Bob', 'Chloé', 'Dimitri'],
    'age': [30, 25, 35, 28],
    'ville': ['Paris', 'Lyon', 'Paris', 'Nice'],
    'salaire': [45000, 38000, 52000, 41000],
})
df

In [ ]:
df.dtypes

In [ ]:
df.shape, df.columns.tolist()

---


## 4. Lecture/écriture de fichiers

Pandas supporte une vingtaine de formats via des fonctions `read_*` / `to_*`.


In [ ]:
# CSV
# df = pd.read_csv('fichier.csv')
# df.to_csv('sortie.csv', index=False)

In [ ]:
# Parquet (rapide, compact)
# df = pd.read_parquet('f.parquet')
# df.to_parquet('out.parquet')

In [ ]:
# Excel
# df = pd.read_excel('f.xlsx', sheet_name='Feuille1')
# df.to_excel('out.xlsx', index=False)

In [ ]:
# JSON
# df = pd.read_json('f.json')

### Options utiles de `read_csv`

- `sep=';'` pour les CSV français ;
- `encoding='utf-8'` par défaut ;
- `parse_dates=['date']` pour convertir en `datetime` ;
- `dtype={'col': 'int32'}` pour forcer le type ;
- `usecols=['a', 'b']` pour ne charger que certaines colonnes ;
- `chunksize=10_000` pour lire par blocs (gros fichiers).


---


## 5. Inspection rapide


In [ ]:
df.head(2)

In [ ]:
df.tail(2)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.sample(2)

In [ ]:
df['ville'].value_counts()

---


## 6. Sélection avec `[]`, `loc`, `iloc`

Trois façons d'accéder aux données :

- `df['col']` — une colonne (`Series`) ;
- `df[['col1', 'col2']]` — plusieurs colonnes (`DataFrame`) ;
- `df.loc[ligne, colonne]` — **par label** ;
- `df.iloc[i, j]` — **par position entière**.


In [ ]:
df['nom']

In [ ]:
df[['nom', 'age']]

In [ ]:
df.loc[0, 'nom']

In [ ]:
df.loc[0:2, ['nom', 'age']]  # slicing inclusif avec loc !

In [ ]:
df.iloc[0, 0]

In [ ]:
df.iloc[0:2, 0:2]  # slicing exclusif avec iloc (comme Python)

**Piège** : `loc` inclut la borne finale, `iloc` ne l'inclut pas.


---


## 7. Filtrage et `query`


In [ ]:
df[df['age'] > 28]

In [ ]:
df[(df['age'] > 28) & (df['ville'] == 'Paris')]

### `query` — syntaxe plus lisible


In [ ]:
df.query('age > 28 and ville == "Paris"')

In [ ]:
df[df['ville'].isin(['Paris', 'Lyon'])]

In [ ]:
df[df['nom'].str.startswith('A')]

---


## 8. Ajouter et supprimer des colonnes


In [ ]:
df2 = df.copy()
df2['age_dans_10_ans'] = df2['age'] + 10
df2

In [ ]:
df2 = df2.assign(impot=lambda d: d['salaire'] * 0.3)
df2

In [ ]:
df2 = df2.drop(columns=['age_dans_10_ans'])
df2

**Astuce** : `assign` renvoie une copie, utile pour chaîner les étapes.


---


## 9. Tri et classement


In [ ]:
df.sort_values('salaire', ascending=False)

In [ ]:
df.sort_values(['ville', 'age'])

In [ ]:
df['salaire'].rank(ascending=False)

---


## 10. Valeurs manquantes (`NaN`)

Les valeurs absentes sont représentées par `NaN` (type float) ou `pd.NA` (nouveau type cohérent, depuis Pandas 1.0).


In [ ]:
d = pd.DataFrame({'a': [1, 2, None, 4], 'b': [5, None, None, 8]})
d

In [ ]:
d.isna()

In [ ]:
d.isna().sum()

In [ ]:
d.dropna()

In [ ]:
d.fillna(0)

In [ ]:
d.fillna({'a': d['a'].mean(), 'b': d['b'].mean()})

### `ffill` et `bfill` pour les séries temporelles


In [ ]:
d.ffill()  # propage la dernière valeur connue

---


## 11. `groupby` et agrégations

Équivalent de `GROUP BY` en SQL. Renvoie un objet paresseux qu'on agrège ensuite.


In [ ]:
df.groupby('ville')['salaire'].mean()

In [ ]:
df.groupby('ville')['salaire'].agg(['mean', 'min', 'max', 'count'])

In [ ]:
df.groupby('ville').agg(salaire_moyen=('salaire', 'mean'), age_max=('age', 'max'))

### `transform` — garder la forme d'origine


In [ ]:
df['salaire_moyen_ville'] = df.groupby('ville')['salaire'].transform('mean')
df

### `apply` sur un groupe — fonction personnalisée


In [ ]:
df.groupby('ville').apply(lambda g: g['salaire'].max() - g['salaire'].min())

---


## 12. `pivot_table` et `melt`


In [ ]:
ventes = pd.DataFrame({
    'produit': ['A', 'B', 'A', 'B', 'A', 'B'],
    'region': ['N', 'N', 'S', 'S', 'E', 'E'],
    'ca': [100, 200, 150, 250, 120, 220],
})
ventes

In [ ]:
ventes.pivot_table(index='produit', columns='region', values='ca')

In [ ]:
ventes.pivot_table(index='produit', columns='region', values='ca', aggfunc='sum', fill_value=0)

### `melt` : de large à long


In [ ]:
wide = pd.DataFrame({
    'annee': [2023, 2024],
    'paris': [100, 110],
    'lyon':  [80, 85],
})
wide.melt(id_vars='annee', var_name='ville', value_name='valeur')

---


## 13. Jointures : `merge`, `concat`, `join`


In [ ]:
gauche = pd.DataFrame({'id': [1, 2, 3], 'nom': ['A', 'B', 'C']})
droite = pd.DataFrame({'id': [2, 3, 4], 'score': [20, 30, 40]})

In [ ]:
gauche.merge(droite, on='id', how='inner')

In [ ]:
gauche.merge(droite, on='id', how='left')

In [ ]:
gauche.merge(droite, on='id', how='outer')

### `concat` pour empiler


In [ ]:
pd.concat([gauche, gauche], ignore_index=True)

In [ ]:
pd.concat([gauche, droite], axis=1)

---


## 14. MultiIndex


In [ ]:
idx = pd.MultiIndex.from_tuples([
    ('Paris', 2023), ('Paris', 2024),
    ('Lyon', 2023), ('Lyon', 2024),
], names=['ville', 'annee'])
pop = pd.Series([2.1, 2.2, 0.5, 0.52], index=idx)
pop

In [ ]:
pop['Paris']

In [ ]:
pop.loc[('Paris', 2024)]

In [ ]:
pop.unstack()

---


## 15. Séries temporelles


In [ ]:
dates = pd.date_range('2024-01-01', periods=10, freq='D')
ts = pd.Series(range(10), index=dates)
ts

In [ ]:
ts['2024-01-03':'2024-01-06']

In [ ]:
ts.resample('2D').sum()

In [ ]:
ts.rolling(window=3).mean()

### Conversion de string en datetime


In [ ]:
pd.to_datetime(['2024-01-15', '2024-02-20'], format='%Y-%m-%d')

---


## 16. `apply` et vectorisation

`apply` est souple mais souvent **lent**. Préférez les opérations vectorisées.


In [ ]:
# Lent : apply ligne par ligne
df['label_lent'] = df.apply(lambda r: r['nom'].upper(), axis=1)
# Rapide : accesseur str vectorisé
df['label_rapide'] = df['nom'].str.upper()
df[['label_lent', 'label_rapide']]

**Règle** : si un accesseur `.str`, `.dt`, ou une opération NumPy existe, elle sera toujours plus rapide que `apply`.


---


## Synthèse

| Notion | À retenir |
|---|---|
| `Series` | colonne étiquetée 1D, dtype uniforme |
| `DataFrame` | table 2D = dict de Series partageant l'index |
| `loc` vs `iloc` | `loc` par label (inclusif), `iloc` par position (exclusif) |
| `query` | filtrage plus lisible qu'avec les masks booléens |
| `NaN` | valeur manquante, gérée par `isna`, `dropna`, `fillna` |
| `groupby` | split-apply-combine ; `agg`, `transform`, `apply` |
| `merge` | jointure SQL (`inner`, `left`, `right`, `outer`) |
| `resample` | agrège par fréquence temporelle (`D`, `M`, `H`...) |
| Vectorisation | préférer `.str`, `.dt`, NumPy à `apply` |


### Règles à retenir

- **Toujours** nommer les colonnes, jamais dépendre des indices.
- **`loc` inclusif, `iloc` exclusif** — piège classique.
- **`SettingWithCopyWarning`** : utiliser `.copy()` ou `.loc[...]` explicite.
- **Vectoriser avant `apply`** : `.str`, `.dt`, NumPy ufuncs.
- **Parquet** plutôt que CSV pour les gros volumes (10x plus rapide).


---


## Exercices

Pour chaque exercice : écrivez une **fonction typée**, testez-la, puis comparez avec la correction. N'ouvrez pas la correction avant d'avoir vraiment cherché.


### Exercice 1 — Moyenne par groupe

Écrivez `moyenne_salaire_par_ville(df)` qui renvoie une `Series` indexée par ville, contenant le salaire moyen.


In [ ]:
import pandas as pd

def moyenne_salaire_par_ville(df: pd.DataFrame) -> pd.Series:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
import pandas as pd

def moyenne_salaire_par_ville(df: pd.DataFrame) -> pd.Series:
    return df.groupby('ville')['salaire'].mean()
```

</details>


### Exercice 2 — Top N

Écrivez `top_n(df, col, n)` qui renvoie les `n` lignes ayant les plus grandes valeurs dans la colonne `col`.


In [ ]:
def top_n(df: pd.DataFrame, col: str, n: int) -> pd.DataFrame:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
def top_n(df: pd.DataFrame, col: str, n: int) -> pd.DataFrame:
    return df.sort_values(col, ascending=False).head(n)
```

</details>


### Exercice 3 — Joindre deux tables

Deux DataFrames `clients` et `commandes` partagent la colonne `client_id`. Écrivez `ca_par_client(clients, commandes)` qui renvoie un DataFrame contenant le nom du client et son chiffre d'affaires total.


In [ ]:
def ca_par_client(clients: pd.DataFrame, commandes: pd.DataFrame) -> pd.DataFrame:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
def ca_par_client(clients: pd.DataFrame, commandes: pd.DataFrame) -> pd.DataFrame:
    ca = commandes.groupby('client_id')['montant'].sum().reset_index(name='ca')
    return clients.merge(ca, on='client_id', how='left').fillna({'ca': 0})
```

</details>


### Exercice 4 — Pivot de ventes

Un DataFrame `v` contient les colonnes `produit`, `mois`, `quantite`. Écrivez `pivot_ventes(v)` qui retourne un tableau avec les produits en lignes, les mois en colonnes, et les quantités en valeurs (0 si absent).


In [ ]:
def pivot_ventes(v: pd.DataFrame) -> pd.DataFrame:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
def pivot_ventes(v: pd.DataFrame) -> pd.DataFrame:
    return v.pivot_table(index='produit', columns='mois',
                          values='quantite', aggfunc='sum', fill_value=0)
```

</details>


### Exercice 5 — Série temporelle : moyenne glissante

Soit une `Series` indexée par date, écrivez `ma(s, k)` qui renvoie la moyenne glissante sur `k` jours.


In [ ]:
def ma(s: pd.Series, k: int) -> pd.Series:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
def ma(s: pd.Series, k: int) -> pd.Series:
    return s.rolling(window=k).mean()
```

</details>


---


## Ressources externes

- [Documentation Pandas](https://pandas.pydata.org/docs/)
- [Pandas Cookbook](https://pandas.pydata.org/docs/user_guide/cookbook.html)
- [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Modern Pandas (Tom Augspurger)](https://tomaugspurger.net/posts/modern-1-intro/)
